<a href="https://colab.research.google.com/github/khanhtran94/crawTTV/blob/master/predicete_vietlot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

# =============================================================
# 1. TẢI DỮ LIỆU THẬT POWER 6/55 TỪ GITHUB
# =============================================================
url = "https://raw.githubusercontent.com/vietvudanh/vietlott-data/main/data/power655.jsonl"

print("Đang tải dữ liệu Power 6/55 từ GitHub...")
df = pd.read_json(url, lines=True)

# Lấy 6 số chính trong mỗi kỳ quay (bỏ qua số đặc biệt thứ 7)
data = [row[:6] for row in df['result']]

num_draws = len(data)
total_numbers = 55  # Power 6/55 gồm các số từ 1 đến 55

print(f"\n=== ĐÃ TẢI THÀNH CÔNG {num_draws} KỲ QUAY ===")
print("5 kỳ gần nhất từ dữ liệu thật:")
for i, d in enumerate(data[-5:], start=num_draws - 4):
    print(f"Kỳ {i}: {sorted(d)}")

# Hàm hỗ trợ tạo 3 vé dựa trên phân phối xác suất
def sample_ticket_from_probs(probabilities, num_tickets=3, temperature=1.0):
    probs = np.clip(probabilities, 1e-8, None)
    scaled_probs = np.exp(np.log(probs) / temperature)
    norm_probs = scaled_probs / np.sum(scaled_probs)

    tickets = []
    for _ in range(num_tickets):
        chosen = np.random.choice(range(1, total_numbers + 1), size=6, replace=False, p=norm_probs)
        tickets.append(sorted(list(chosen)))
    return tickets


# =============================================================
# 2. STRATEGY 2: LSTM (PYTORCH)
# =============================================================
class LSTMModel(nn.Module):
    def __init__(self, input_size=55, hidden_size=64):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, input_size)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])
        return out

seq_len = 5
X_seq, y_seq = [], []
for i in range(seq_len, len(data)):              # Iterate up to the second to last element
    seq_in = [np.eye(total_numbers)[np.array(draw) - 1].sum(axis=0) for draw in data[i - seq_len:i]]
    seq_out = np.eye(total_numbers)[np.array(data[i]) - 1].sum(axis=0)
    X_seq.append(seq_in)
    y_seq.append(seq_out)

X_seq_t = torch.tensor(np.array(X_seq), dtype=torch.float32)
y_seq_t = torch.tensor(np.array(y_seq), dtype=torch.float32)

lstm = LSTMModel(input_size=total_numbers)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(lstm.parameters(), lr=0.008)

print("\nĐang huấn luyện LSTM...")
lstm.train()
for epoch in range(60):
    optimizer.zero_grad()
    outputs = lstm(X_seq_t)
    loss = criterion(outputs, y_seq_t)
    loss.backward()
    optimizer.step()

last_seq = [np.eye(total_numbers)[np.array(draw) - 1].sum(axis=0) for draw in data[-seq_len:]]
last_seq_t = torch.tensor(np.array([last_seq]), dtype=torch.float32)

lstm.eval()
with torch.no_grad():
    logits = lstm(last_seq_t)
    lstm_probs = torch.sigmoid(logits).numpy()[0]

print("\n" + "="*40)
print("  STRATEGY 2: LSTM - 3 TICKETS")
print("="*40)
for idx, temp in enumerate([0.7, 1.0, 1.3], start=1):
    ticket = sample_ticket_from_probs(lstm_probs, num_tickets=1, temperature=temp)[0]
    print(f"Ticket {idx} (Temp {temp:.1f}): {ticket}")

Đang tải dữ liệu Power 6/55 từ GitHub...

=== ĐÃ TẢI THÀNH CÔNG 1402 KỲ QUAY ===
5 kỳ gần nhất từ dữ liệu thật:
Kỳ 1398: [24, 27, 32, 36, 38, 47]
Kỳ 1399: [6, 11, 25, 27, 37, 45]
Kỳ 1400: [4, 7, 11, 18, 22, 25]
Kỳ 1401: [1, 3, 9, 11, 41, 46]
Kỳ 1402: [1, 13, 23, 25, 26, 28]

Đang huấn luyện LSTM...

  STRATEGY 2: LSTM - 3 TICKETS
Ticket 1 (Temp 0.7): [np.int64(6), np.int64(22), np.int64(26), np.int64(34), np.int64(38), np.int64(47)]
Ticket 2 (Temp 1.0): [np.int64(16), np.int64(22), np.int64(32), np.int64(41), np.int64(47), np.int64(54)]
Ticket 3 (Temp 1.3): [np.int64(3), np.int64(7), np.int64(14), np.int64(15), np.int64(25), np.int64(40)]


### Giải thích tham số Temperature trong LSTM (dựa trên hàm `sample_ticket_from_probs`)

Trong ngữ cảnh của mã bạn cung cấp, `Temperature` là một tham số được sử dụng trong quá trình lấy mẫu (sampling) để điều chỉnh mức độ 'ngẫu nhiên' hoặc 'sự tự tin' trong việc chọn các con số từ phân phối xác suất được dự đoán bởi mô hình LSTM.

Phân phối xác suất (`probabilities`) từ mô hình LSTM cho biết khả năng xuất hiện của mỗi con số từ 1 đến 55. Nhiệt độ (Temperature) được áp dụng để làm 'sắc nét' hoặc 'làm mềm' phân phối xác suất này trước khi chọn các con số.

Công thức bạn đang dùng để tính `scaled_probs` là:

`scaled_probs = np.exp(np.log(probs) / temperature)`

Đây là cách nó hoạt động:

*   **Temperature thấp (ví dụ: 0.7):**
    *   Khi `temperature` nhỏ hơn 1, nó sẽ làm cho các xác suất cao trở nên nổi bật hơn và các xác suất thấp trở nên thấp hơn nữa một cách tương đối. Điều này có nghĩa là mô hình sẽ có xu hướng chọn các con số mà nó 'tự tin' nhất về khả năng xuất hiện của chúng.
    *   Kết quả là các vé được tạo ra sẽ có ít sự đa dạng hơn và tập trung vào các con số có xác suất cao nhất. Điều này có thể dẫn đến việc các vé 'giống nhau' hơn hoặc tập trung vào một tập hợp nhỏ các con số 'nóng'.
    *   Trong bối cảnh tìm 'vé trúng nhiều số hơn', một `temperature` thấp có thể hữu ích nếu mô hình của bạn *thực sự* tốt trong việc xác định các con số có khả năng trúng cao. Nó sẽ khai thác tối đa những dự đoán 'tốt nhất' của mô hình.

*   **Temperature bằng 1 (mặc định):**
    *   Các xác suất được sử dụng gần như trực tiếp (sau khi chuẩn hóa lại).
    *   Đây là một sự cân bằng giữa việc chọn các số có xác suất cao và vẫn giữ một mức độ ngẫu nhiên nhất định.

*   **Temperature cao (ví dụ: 1.3):**
    *   Khi `temperature` lớn hơn 1, nó sẽ làm cho phân phối xác suất trở nên 'mềm hơn' hoặc 'phẳng hơn'. Điều này có nghĩa là sự khác biệt giữa các xác suất cao và thấp bị giảm đi, làm tăng cơ hội chọn các con số có xác suất thấp hơn.
    *   Kết quả là các vé được tạo ra sẽ đa dạng hơn, bao gồm nhiều con số ít có khả năng xuất hiện hơn theo dự đoán của mô hình. Điều này tạo ra sự khám phá nhiều hơn trong không gian số.
    *   Trong bối cảnh tìm 'vé trúng nhiều số hơn', một `temperature` cao có thể hữu ích nếu bạn muốn khám phá nhiều tổ hợp số khác nhau, đặc biệt nếu bạn nghĩ rằng mô hình có thể bỏ lỡ một số con số tiềm năng nhưng có xác suất thấp hơn. Nó có thể giúp bạn tránh bị 'kẹt' với chỉ những con số mà mô hình 'tin tưởng' nhất.

### Làm thế nào để điều chỉnh Temperature để có thể 'vé trúng nhiều số hơn'?

Không có một giá trị `temperature` 'thần kỳ' nào đảm bảo 'vé trúng nhiều số hơn', vì bản chất xổ số là ngẫu nhiên và mô hình chỉ đưa ra dự đoán dựa trên dữ liệu lịch sử.

Tuy nhiên, bạn có thể thử nghiệm như sau:

1.  **Nếu bạn tin tưởng vào mô hình của mình:**
    *   Thử các giá trị `temperature` **thấp hơn** (ví dụ: 0.5, 0.7) để ưu tiên các con số có xác suất cao nhất mà mô hình dự đoán. Điều này sẽ tạo ra các vé ít ngẫu nhiên hơn và tập trung vào những con số 'tốt nhất' theo mô hình.
    *   **Lưu ý:** Điều này có thể dẫn đến việc các vé của bạn trở nên tương tự nhau nếu mô hình luôn đưa ra các dự đoán mạnh mẽ cho cùng một tập hợp các số.

2.  **Nếu bạn muốn đa dạng hóa và khám phá nhiều hơn:**
    *   Thử các giá trị `temperature` **cao hơn** (ví dụ: 1.2, 1.5) để cho phép nhiều con số có xác suất thấp hơn được chọn. Điều này sẽ tạo ra các vé đa dạng hơn và ít phụ thuộc vào các dự đoán 'tự tin' nhất của mô hình.
    *   **Lưu ý:** Điều này có thể bao gồm các con số thực sự có xác suất thấp, nhưng đôi khi những con số 'bất ngờ' lại xuất hiện.

3.  **Cách tiếp cận đề xuất:**
    *   Bạn nên tiếp tục sử dụng nhiều giá trị `temperature` khác nhau (như bạn đã làm với 0.7, 1.0, 1.3) để tạo ra nhiều loại vé. Điều này giúp bạn có được cả những vé tập trung vào dự đoán 'mạnh mẽ' của mô hình và những vé khám phá các khả năng khác.
    *   Hãy nhớ rằng, mục tiêu của `temperature` là để kiểm soát sự cân bằng giữa việc *khai thác* (chọn những thứ mà mô hình tin là tốt nhất) và *khám phá* (thử những điều mới mẻ). Trong một trò chơi may rủi như xổ số, việc khám phá một chút có thể là chiến lược tốt để bao quát nhiều khả năng hơn.